In [1]:
import pandas as pd
import numpy as np
import os
import requests
import time
import random

# Configuration
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
REAL_CATS_DATA_DIR = os.path.join(PROJECT_ROOT, 'Real_Cats_data')
PATH_BENIGN = os.path.join(REAL_CATS_DATA_DIR, 'BB.tsv')
PATH_CRIMINAL = os.path.join(REAL_CATS_DATA_DIR, 'CB.tsv')

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Directory: {REAL_CATS_DATA_DIR}")

Project Root: d:\Projects\final_project
Data Directory: d:\Projects\final_project\Real_Cats_data


In [2]:
def split_criminal_wallet(input_path=f"{REAL_CATS_DATA_DIR}/CB.tsv", output_dir=f"{REAL_CATS_DATA_DIR}/"):

    df = pd.read_csv(input_path, sep='\t', low_memory=False)
    behavioral_wallets = df[
        (df["transaction_number"] > 0) |
        (df["total_received_BTC"] > 0) |
        (df["total_sent_BTC"] > 0)
    ].copy()

    non_behavioral_wallets = df[
        (df["transaction_number"] == 0) &
        (df["total_received_BTC"] == 0) &
        (df["total_sent_BTC"] == 0)
    ].copy()

    print("Total wallets:", len(df))
    print("Behavioral:", len(behavioral_wallets))
    print("Non-behavioral:", len(non_behavioral_wallets))

    behavioral_wallets.to_csv(f"{output_dir}/wallets_behavioral.tsv", sep='\t', index=False)
    non_behavioral_wallets.to_csv(f"{output_dir}/wallets_non_behavioral.tsv", sep='\t', index=False)

    print("Saved wallets_behavioral and non-behavioral wallets to", output_dir)
    return behavioral_wallets, non_behavioral_wallets

split_criminal_wallet()

Total wallets: 90597
Behavioral: 40032
Non-behavioral: 50565
Saved wallets_behavioral and non-behavioral wallets to d:\Projects\final_project\Real_Cats_data/


(                                          address              label  \
 0               111KvKxkeia8NeKMzqEDqnGm1v49Ncp3j     Blackmail Scam   
 1              11212qhtzpz2SugohG3xk1FCWY9U5TG8mg         Ransomware   
 4              1122NYbAT2KkZDZ5TFvGy4D2Ut7eYfx4en         Ransomware   
 6              1123Hubtw2CXGqFKvvozUcbWe6SbTT5ydg         Ransomware   
 14             1128Ev6iMRQ25SuGAnrekSqbW8aQpX5PZQ    Investment Scam   
 ...                                           ...                ...   
 90583  bc1qzzaxpxts27jxcq842fazy7sqvuese0njt5qrfv              Other   
 90585  bc1qzzdgdj3tzv9fsww3p9e2jcm5pzemh9uawv0cus  Social Media Scam   
 90586  bc1qzznxeqlcvzey9nkaeaffv64pna8n8tpnr8l6nn               Hack   
 90590  bc1qzzx3esv0a0he9nkmjxza3alfycgx4jhgqullz9      Giveaway Scam   
 90591  bc1qzzx8fmg85w33608lqwh7unh6yk9nxyjsgxy2pq              Other   
 
         balance  total_received_BTC  total_sent_BTC  total_received_USD  \
 0           0.0            180000.0        18

In [2]:
def fetch_wallet_transactions(address, first_time=None, last_time=None, max_retries=3):
    """
    Fetch all transactions for a single wallet address from mempool.space API.
    Returns a list of edge dictionaries (source, target, weight, timestamp, direction).
    """
    edges_list = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Parse time window
    try:
        start_ts = pd.to_datetime(first_time).timestamp() if first_time else 0
        end_ts = pd.to_datetime(last_time).timestamp() if last_time else 9999999999
    except:
        start_ts, end_ts = 0, 9999999999
    
    retries = 0
    while retries <= max_retries:
        try:
            url = f"https://mempool.space/api/address/{address}/txs"
            r = requests.get(url, headers=headers, timeout=15)
            
            if r.status_code == 200:
                txs = r.json()
                
                for tx in txs:
                    # Check if transaction is confirmed
                    if not tx.get('status', {}).get('confirmed'):
                        continue
                        
                    tx_time = tx['status']['block_time']
                    
                    # Check time window
                    if tx_time < start_ts or tx_time > end_ts:
                        continue
                    
                    # Check if wallet is sender (appears in inputs)
                    is_sender = False
                    for inp in tx.get('vin', []):
                        if inp.get('prevout', {}).get('scriptpubkey_address') == address:
                            is_sender = True
                            break
                    
                    # If sender, create edges to all recipients
                    if is_sender:
                        for out in tx.get('vout', []):
                            recipient = out.get('scriptpubkey_address')
                            amount = out.get('value', 0)  # Satoshis
                            
                            if recipient and recipient != address:
                                edges_list.append({
                                    'source': address,
                                    'target': recipient,
                                    'weight': amount,
                                    'timestamp': tx_time,
                                    'direction': 'outgoing',
                                    'txid': tx.get('txid', '')
                                })
                    
                    # Check if wallet is receiver (appears in outputs)
                    amount_received = 0
                    is_receiver = False
                    for out in tx.get('vout', []):
                        if out.get('scriptpubkey_address') == address:
                            amount_received += out.get('value', 0)
                            is_receiver = True
                    
                    # If receiver, create edges from all senders
                    if is_receiver:
                        for inp in tx.get('vin', []):
                            sender = inp.get('prevout', {}).get('scriptpubkey_address')
                            
                            if sender and sender != address:
                                edges_list.append({
                                    'source': sender,
                                    'target': address,
                                    'weight': amount_received,
                                    'timestamp': tx_time,
                                    'direction': 'incoming',
                                    'txid': tx.get('txid', '')
                                })
                break  # Success, exit retry loop
            
            elif r.status_code == 429:
                retries += 1
                wait_time = 5 * retries  # Exponential backoff
                print(f"      Rate limited! Waiting {wait_time}s... (retry {retries}/{max_retries})")
                time.sleep(wait_time)
            
            else:
                print(f"      HTTP {r.status_code} for {address}")
                break  # Non-retryable error
        
        except requests.exceptions.Timeout:
            retries += 1
            print(f"      Timeout! Retry {retries}/{max_retries}")
            time.sleep(2)
        except Exception as e:
            print(f"      Error fetching {address}: {e}")
            break
    
    return edges_list

In [3]:
def fetch_random_wallets(input_path, label, output_path, n_wallets=1000, only_with_edges=False):
    """
    Fetch transactions for n random wallets total, skipping already processed ones.
    
    Args:
        only_with_edges: If True, only count wallets that have at least 1 edge (transaction).
                        Wallets with no edges are marked as processed but don't count toward n_wallets.
    """
    
    print(f"\n{'='*60}")
    print(f"Fetching up to {n_wallets} total {label.upper()} wallets")
    if only_with_edges:
        print(f"(Only counting wallets WITH transactions)")
    print(f"{'='*60}\n")
    
    # Load wallet data
    df = pd.read_csv(input_path, sep='\t', low_memory=False)
    df = df[df['address'].notna()].copy()
    
    # Load already processed wallets
    progress_file = output_path.replace('.csv', '_progress.txt')
    already_fetched = set()
    
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            already_fetched = set(line.strip() for line in f if line.strip())
    
    # Count wallets with edges (if only_with_edges mode)
    if only_with_edges and os.path.exists(output_path):
        edges_df = pd.read_csv(output_path)
        # Count unique source wallets that match our label
        wallets_with_edges = set(edges_df[edges_df['wallet_label'] == label]['source'].unique()) | \
                           set(edges_df[edges_df['wallet_label'] == label]['target'].unique())
        wallets_with_edges = wallets_with_edges & already_fetched  # Only count ones we processed
        successful_wallets = len(wallets_with_edges)
    else:
        successful_wallets = len(already_fetched)
    
    print(f"Already processed: {len(already_fetched)} wallets")
    if only_with_edges:
        print(f"Wallets with edges: {successful_wallets}")
    
    # Check if we've already reached the target
    if successful_wallets >= n_wallets:
        print(f"Already have {successful_wallets} wallets with edges (target: {n_wallets}). Nothing to do!")
        return
    
    # Filter out already processed
    df = df[~df['address'].isin(already_fetched)].copy()
    print(f"Available wallets (not yet processed): {len(df)}")
    
    if len(df) == 0:
        print("No more wallets available to process!")
        return
    
    # Shuffle the dataframe for random sampling
    df_shuffled = df.sample(frac=1, random_state=random.randint(1, 10000))
    
    # Prepare output file
    file_exists = os.path.exists(output_path)
    if not file_exists:
        pd.DataFrame(columns=['source', 'target', 'weight', 'timestamp', 'direction', 'txid', 'wallet_label']).to_csv(
            output_path, index=False, mode='w'
        )
    
    start_time = time.time()
    wallets_fetched = 0
    wallets_with_edges_count = successful_wallets
    
    for idx, (i, row) in enumerate(df_shuffled.iterrows()):
        # Stop if we've reached our target
        if wallets_with_edges_count >= n_wallets:
            break
            
        address = row['address']
        first_time = row.get('first_time', None)
        last_time = row.get('last_time', None)
        
        elapsed = time.time() - start_time
        rate = wallets_fetched / elapsed if elapsed > 0 else 0
        
        if only_with_edges:
            print(f"[{wallets_with_edges_count}/{n_wallets} with edges] {address[:25]}... | {elapsed/60:.1f}m elapsed")
        else:
            current_total = len(already_fetched) + wallets_fetched + 1
            print(f"[{current_total}/{n_wallets}] {address[:25]}... | {elapsed/60:.1f}m elapsed")
        
        edges = fetch_wallet_transactions(address, first_time, last_time)
        
        if edges:
            print(f"      Found {len(edges)} edges ✓")
            for edge in edges:
                edge['wallet_label'] = label
            pd.DataFrame(edges).to_csv(output_path, index=False, mode='a', header=False)
            wallets_with_edges_count += 1
        else:
            print(f"      No edges found (skipping)")
        
        # Mark as processed (regardless of whether it had edges)
        with open(progress_file, 'a') as f:
            f.write(f"{address}\n")
        
        wallets_fetched += 1
        time.sleep(1)
    
    print(f"\n{'='*60}")
    if only_with_edges:
        print(f"DONE! Now have {wallets_with_edges_count} {label} wallets with edges (checked {wallets_fetched} this run)")
    else:
        print(f"DONE! Now have {len(already_fetched) + wallets_fetched} {label} wallets total")
    print(f"{'='*60}\n")

In [4]:
# Fetch 1000 random benign wallets
bb_output = os.path.join(REAL_CATS_DATA_DIR, 'bb_transactions.csv')
fetch_random_wallets(PATH_BENIGN, 'benign', bb_output, n_wallets=1000)


Fetching up to 1000 total BENIGN wallets

Already processed: 1000 wallets
Already have 1000 wallets with edges (target: 1000). Nothing to do!


In [5]:
# Fetch 1000 random criminal wallets (only ones with transactions)
cb_output = os.path.join(REAL_CATS_DATA_DIR, 'cb_transactions.csv')
fetch_random_wallets(PATH_CRIMINAL, 'criminal', cb_output, n_wallets=1000, only_with_edges=True)


Fetching up to 1000 total CRIMINAL wallets
(Only counting wallets WITH transactions)

Already processed: 0 wallets
Wallets with edges: 0
Available wallets (not yet processed): 90597
[0/1000 with edges] bc1q2jufjr65aexju20w6fvqw... | 0.0m elapsed
      No edges found (skipping)
[0/1000 with edges] bc1qppx0tqcr8yg3wt5xd8z9j... | 0.0m elapsed
      No edges found (skipping)
[0/1000 with edges] 1MvVjiJPHHKT9esMFK4cfGM7k... | 0.0m elapsed
      Found 1 edges ✓
[1/1000 with edges] 1AM7Pmsdq5dViz9xkvpAQuCcF... | 0.1m elapsed
      No edges found (skipping)
[1/1000 with edges] bc1qntl97rlmp88m9yurt76wl... | 0.1m elapsed
      Found 1 edges ✓
[2/1000 with edges] 1Kg9onSxbyTmQVGiwpGvJ1bsD... | 0.1m elapsed
      No edges found (skipping)
[2/1000 with edges] bc1qpqvsjp95kzc67s3w7e3q3... | 0.1m elapsed
      Found 21 edges ✓
[3/1000 with edges] 15GqSWnxEFZezUCcGjhBMknA1... | 0.2m elapsed
      Found 1 edges ✓
[4/1000 with edges] 1F4stC1h96tgYpfetTkc44ixL... | 0.2m elapsed
      No edges found (sk